# 07: Grader predictions

Runs a trained grader (the `best.pt` of notebook 06) once over every image that evaluation and
calibration need: EyePACS `val`, `calib` and `test_internal`, and all external images (APTOS,
IDRiD, DDR), quality rejects and ungradable images included. Saves the probabilities and the
pooled image embeddings; metrics with confidence intervals are computed from these files
locally (`python -m retinaguard.train.evaluate`). A quick look at the metrics is printed here.

**Settings** (right-hand panel): Accelerator **GPU T4 x1**, Internet **On**.
**Inputs** (*Add Input*):
- *Notebooks* → *Your Work* → **01**, **02**, **03**, **04** (processed image shards of all
  sources) and **06: Train the DR grader** (the run to evaluate)
- *Datasets* → *Your Datasets* → **retinaguard-manifest**

**Run:** set `RUN_NAME` below, then *Save Version* → *Save & Run All (Commit)*. About 10 min.

**Output** (`/kaggle/working/predictions/<run name>/`): `predictions.parquet`,
`embeddings.npy`, `quick_metrics.csv`.

> RetinaGuard is a research prototype, **not a medical device**.

In [ ]:
REPO_URL = "https://github.com/FidaMarzouk/retinaguard.git"
GIT_REF = "main"
RUN_NAME = "grader-convnext_tiny"  # the run folder written by notebook 06

In [ ]:
# Kaggle's preinstalled albumentations may predate the 2.x API the package imports.
EXTRAS = '"pandera>=0.20" "hydra-core>=1.3" "timm>=1.0" "albumentations>=2.0"'
%pip install -q "retinaguard @ git+{REPO_URL}@{GIT_REF}" {EXTRAS}

In [ ]:
# --- 1. Inputs -------------------------------------------------------------------
import re
import subprocess
from pathlib import Path

import pandas as pd
import torch

from retinaguard import __version__

INPUT_ROOT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working/predictions") / RUN_NAME
checkpoints = sorted(INPUT_ROOT.glob(f"**/runs/{RUN_NAME}/best.pt"))
assert checkpoints, f"runs/{RUN_NAME}/best.pt not found: attach the output of notebook 06"
manifests = sorted(INPUT_ROOT.glob("**/manifest.parquet"))
assert manifests, "manifest.parquet not found: attach the retinaguard-manifest dataset"
shard = re.compile(r"^(eyepacs|aptos|idrid|ddr)_\d{3}\.zip$")
shard_dirs = sorted({p.parent for p in INPUT_ROOT.glob("**/*.zip") if shard.match(p.name)})
sources = sorted({p.name.split("_")[0] for d in shard_dirs for p in d.glob("*.zip")})
assert sources == ["aptos", "ddr", "eyepacs", "idrid"], f"shards found for {sources} only"

git_sha = subprocess.run(
    ["git", "ls-remote", REPO_URL, GIT_REF], capture_output=True, text=True
).stdout.split("	")[0]
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
assert torch.cuda.is_available(), (
    f"no GPU (torch {torch.__version__}): set Accelerator to GPU in the notebook settings, "
    "then save a new version"
)
print(f"retinaguard {__version__} @ {git_sha[:10]}, GPU {gpu}")
print("checkpoint:", checkpoints[0])

In [ ]:
# --- 2. Predict ------------------------------------------------------------------
import time

from retinaguard.train.predict import main as predict

start = time.perf_counter()
predict(
    [
        f"--checkpoint={checkpoints[0]}",
        f"--manifest={manifests[0]}",
        "--shard-dirs",
        *map(str, shard_dirs),
        f"--out-dir={OUT_DIR}",
    ]
)
print(f"{(time.perf_counter() - start) / 60:.1f} min")

In [ ]:
# --- 3. Quick look (200 bootstrap resamples; the reported numbers use 2,000) ---------
from retinaguard.train.evaluate import evaluate

manifest = pd.read_parquet(manifests[0])
predictions = pd.read_parquet(OUT_DIR / "predictions.parquet")
table = evaluate(manifest, predictions, n_resamples=200)
table.to_csv(OUT_DIR / "quick_metrics.csv")
pd.set_option("display.width", 200)
columns = [
    "images",
    "referable",
    "auroc",
    "auroc_low",
    "auroc_high",
    "auprc",
    "sens_at_90_spec",
    "spec_at_95_sens",
    "quadratic_kappa",
]
print(table[columns].round(3).to_string())

**After the version finishes:** download `predictions.parquet` and `embeddings.npy` from the
**Output** tab into `data/interim/predictions/<run name>/`, and send the table of cell 3.